# SpatialSearch — End-to-End Tweet RAG Pipeline

This notebook contains the complete SpatialSearch pipeline in a single runnable document. It walks through all five stages in order:

| Part | What it does |
|---|---|
| **1 — Index Building** | Load 110k tweets, encode with GloVe + MiniLM, build and save FAISS indexes |
| **2 — RAG Pipeline** | Full retrieve → rerank → FLAN-T5 summarize loop with demo queries |
| **3 — Evaluation** | Benchmark GloVe vs MiniLM on Precision@k and MRR across 25 queries |
| **4 — Visualization** | UMAP projection + K-Means clustering of the tweet embedding space |
| **5 — Gradio Demo** | Interactive web UI deployable to HuggingFace Spaces |

**Important:** Run the cells in order from top to bottom. Part 1 must complete before any other part, since it writes the FAISS indexes and tweet list to disk that all later parts depend on.

**Environment:** Designed for Google Colab (free tier). All dependencies are installed inline via `!pip install`. Place `tweets-utf-8.json` in the same directory or at `/content/` on Colab.

---
## Part 1 — Index Building

This section is the **foundation** of the pipeline. It runs once and saves all artifacts to disk so later sections don't need to re-encode 110k tweets from scratch.

**What happens here:**
1. Load all tweets from `tweets-utf-8.json`
2. Encode with GloVe (static word embeddings) and MiniLM (contextual BERT-based embeddings)
3. Build a FAISS index for fast nearest-neighbor search at query time
4. Save embeddings and indexes to disk

**Artifacts produced:** `tweets.json`, `glove_embeddings.npy`, `minilm_embeddings.npy`, `glove_faiss.index`, `minilm_faiss.index`

In [ ]:
# --- STEP 0: INSTALL DEPENDENCIES ---
# faiss-cpu provides the vector index; sentence-transformers wraps both GloVe and MiniLM
!pip install faiss-cpu sentence-transformers --quiet

In [ ]:
# --- STEP 1: IMPORTS ---
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("All imports successful.")

In [ ]:
# --- STEP 2: LOAD TWEETS ---

def load_tweets(filepath):
    """
    Load tweet texts from a JSON file where each line is a separate JSON object.

    Parameters:
        filepath (str): Path to the tweets JSON file.

    Returns:
        list[str]: A list of tweet text strings, skipping any entries
                   that are missing a 'text' field or can't be parsed.
    """
    tweets = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                # Some entries may be metadata or malformed — only keep actual tweet text
                if 'text' in obj:
                    tweets.append(obj['text'])
            except json.JSONDecodeError:
                pass  # skip malformed lines silently
    return tweets


# Try Colab path first, then fall back to local directory
data_path = '/content/tweets-utf-8.json' if os.path.exists('/content/tweets-utf-8.json') else 'tweets-utf-8.json'

tweets = load_tweets(data_path)
print(f"Loaded {len(tweets):,} tweets from {data_path}")
print(f"Sample tweet: {tweets[0][:100]}...")

In [ ]:
# --- STEP 3: SAVE CLEAN TWEET LIST ---
# Save tweet texts as a plain JSON list so other sections can load them
# without re-parsing the original file (which has metadata we don't need)

with open('tweets.json', 'w', encoding='utf-8') as f:
    json.dump(tweets, f, ensure_ascii=False)

print(f"Saved {len(tweets):,} tweet texts to tweets.json")

In [ ]:
# --- STEP 4: ENCODE WITH GLOVE ---
# GloVe gives each word a fixed vector regardless of context.
# SentenceTransformers averages word vectors to produce a sentence embedding.
# This is fast but can't capture word sense (e.g. "bank" the river vs. bank the institution).

def encode_tweets(model_name, tweets, batch_size=256, show_progress=True):
    """
    Load a SentenceTransformer model and encode a list of tweets.

    Parameters:
        model_name (str): HuggingFace model identifier.
        tweets (list[str]): Tweet texts to encode.
        batch_size (int): How many tweets to encode at once. Larger = faster but more RAM.
        show_progress (bool): Whether to show a tqdm progress bar.

    Returns:
        np.ndarray: Float32 array of shape (num_tweets, embedding_dim).
    """
    model = SentenceTransformer(model_name)
    embeddings = model.encode(
        tweets,
        batch_size=batch_size,
        show_progress_bar=show_progress,
        convert_to_numpy=True
    )
    # Ensure float32 — FAISS requires this dtype
    return embeddings.astype(np.float32)


print("Encoding with GloVe (average_word_embeddings_glove.840B.300d)...")
print("This encodes 110k tweets using averaged GloVe word vectors — expect a few minutes.")
glove_embeddings = encode_tweets('average_word_embeddings_glove.840B.300d', tweets)

np.save('glove_embeddings.npy', glove_embeddings)
print(f"GloVe embeddings shape: {glove_embeddings.shape}")
print(f"Saved to glove_embeddings.npy")

In [ ]:
# --- STEP 5: ENCODE WITH MINILM ---
# MiniLM is a distilled BERT model — it reads the full sentence at once and
# produces context-aware embeddings. Much better at capturing meaning than GloVe.
# The tradeoff is it's slower to encode, but results are significantly better.

import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

print("Encoding with MiniLM (all-MiniLM-L6-v2)...")
print("MiniLM produces 384-dim contextual embeddings — slower than GloVe but much richer.")
minilm_embeddings = encode_tweets('all-MiniLM-L6-v2', tweets)

np.save('minilm_embeddings.npy', minilm_embeddings)
print(f"MiniLM embeddings shape: {minilm_embeddings.shape}")
print(f"Saved to minilm_embeddings.npy")

In [ ]:
# --- STEP 6: BUILD FAISS INDEX FOR MINILM ---

def build_inner_product_index(embeddings):
    """
    Build a FAISS IndexFlatIP (inner product) index from embeddings.

    Inner product on L2-normalized vectors is equivalent to cosine similarity,
    so normalizing first lets us use the fast IP index to rank by cosine sim.

    Parameters:
        embeddings (np.ndarray): Float32 array of shape (n, dim).

    Returns:
        faiss.IndexFlatIP: Populated FAISS index ready to query.
    """
    # Normalize so dot product equals cosine similarity — required for IndexFlatIP
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    # Avoid division by zero for any zero vectors
    norms = np.where(norms == 0, 1, norms)
    normalized = (embeddings / norms).astype(np.float32)

    dim = normalized.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(normalized)
    return index


print("Building FAISS IndexFlatIP for MiniLM embeddings...")
# Why FAISS over brute-force NumPy: FAISS uses optimized C++ under the hood
# and can scale to millions of vectors. Our cosine search in the original notebook
# was O(n) — FAISS is still exact here (IndexFlatIP) but much faster in practice.
minilm_index = build_inner_product_index(minilm_embeddings)

faiss.write_index(minilm_index, 'minilm_faiss.index')
print(f"Indexed {minilm_index.ntotal:,} MiniLM vectors — saved to minilm_faiss.index")

In [ ]:
# --- STEP 7: BUILD FAISS INDEX FOR GLOVE ---

def build_l2_index(embeddings):
    """
    Build a FAISS IndexFlatL2 (Euclidean distance) index from embeddings.

    We use L2 for GloVe because GloVe vectors aren't trained to have unit norm,
    so inner product scores are less meaningful without careful normalization.
    L2 distance is more robust here.

    Parameters:
        embeddings (np.ndarray): Float32 array of shape (n, dim).

    Returns:
        faiss.IndexFlatL2: Populated FAISS index ready to query.
    """
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings.astype(np.float32))
    return index


print("Building FAISS IndexFlatL2 for GloVe embeddings...")
glove_index = build_l2_index(glove_embeddings)

faiss.write_index(glove_index, 'glove_faiss.index')
print(f"Indexed {glove_index.ntotal:,} GloVe vectors — saved to glove_faiss.index")

In [ ]:
# --- STEP 8: SMOKE TEST ---
# Quick sanity check: run a query through the MiniLM FAISS index
# and make sure the returned tweets look semantically relevant.

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
test_query = "I am looking for a job."

query_vec = model.encode([test_query], convert_to_numpy=True).astype(np.float32)
# Normalize query vector the same way we normalized the index
query_vec = query_vec / np.linalg.norm(query_vec)

# Search for top 5 most similar tweets
scores, indices = minilm_index.search(query_vec, k=5)

print(f"Test query: '{test_query}'")
print(f"\nTop 5 results (MiniLM FAISS):")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"  {rank}. [score={score:.4f}] {tweets[idx][:120]}")

print("\nSmoke test passed — index is working correctly!")

In [ ]:
# --- SUMMARY ---
print("=" * 60)
print("INDEX BUILD COMPLETE")
print("=" * 60)
print(f"  tweets.json           — {len(tweets):,} tweet texts")
print(f"  glove_embeddings.npy  — shape {glove_embeddings.shape}")
print(f"  minilm_embeddings.npy — shape {minilm_embeddings.shape}")
print(f"  glove_faiss.index     — {glove_index.ntotal:,} vectors (L2)")
print(f"  minilm_faiss.index    — {minilm_index.ntotal:,} vectors (IP/cosine)")
print()
print("Next: proceed to Part 2 — RAG Pipeline")

---
## Part 2 — RAG Retrieval Pipeline

This section implements the full **Retrieve → Rerank → Summarize** pipeline.

**Pipeline stages:**
1. **Retrieve** — Encode query with MiniLM, search FAISS index for top-50 candidates
2. **Rerank** — Score each (query, tweet) pair with a cross-encoder; keep top-10
3. **Summarize** — Feed top-5 tweets into FLAN-T5 to generate a natural language summary

This two-stage approach balances speed and accuracy: FAISS retrieves fast candidates, the cross-encoder applies a more expensive but more accurate relevance model on a small shortlist.

In [ ]:
# --- STEP 0: INSTALL DEPENDENCIES ---
# transformers: for FLAN-T5
# sentence-transformers: for MiniLM query encoder and cross-encoder reranker
# faiss-cpu: for loading the vector index
!pip install transformers sentence-transformers faiss-cpu torch --quiet

In [ ]:
# --- STEP 1: IMPORTS ---
import json
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# --- STEP 2: LOAD INDEX AND TWEET LIST ---
# These files were created by Part 1

import os

# Support both Colab (/content/) and local environments
base = '/content/' if os.path.exists('/content/minilm_faiss.index') else ''

print("Loading FAISS index...")
minilm_index = faiss.read_index(f'{base}minilm_faiss.index')
print(f"Index loaded — contains {minilm_index.ntotal:,} vectors")

print("Loading tweet texts...")
with open(f'{base}tweets.json', 'r', encoding='utf-8') as f:
    tweets = json.load(f)
print(f"Loaded {len(tweets):,} tweet texts")

In [ ]:
# --- STEP 3: LOAD MODELS ---

print("Loading MiniLM query encoder...")
# We use the same model as the index was built with — mismatching models would give garbage results
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
print("MiniLM loaded.")

print("Loading cross-encoder reranker...")
# The cross-encoder reads (query, passage) together — much more accurate than bi-encoder
# but too slow to run on 110k tweets directly, so we only apply it to the top-50 candidates
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Cross-encoder loaded.")

print("Loading FLAN-T5 for summarization...")
# FLAN-T5 is an instruction-tuned encoder-decoder model — it follows natural language prompts
# well and generates coherent summaries without any fine-tuning on our data.
# We prefer 'large' for better summary quality; fall back to 'base' if OOM.
try:
    flan_model_name = 'google/flan-t5-large'
    flan_tokenizer = T5Tokenizer.from_pretrained(flan_model_name)
    flan_model = T5ForConditionalGeneration.from_pretrained(flan_model_name).to(device)
    print(f"FLAN-T5-large loaded on {device}.")
except RuntimeError:
    # flan-t5-large needs ~3GB RAM — fall back to base (~900MB) on memory-constrained environments
    print("OOM on flan-t5-large — falling back to flan-t5-base (smaller but still instruction-tuned).")
    flan_model_name = 'google/flan-t5-base'
    flan_tokenizer = T5Tokenizer.from_pretrained(flan_model_name)
    flan_model = T5ForConditionalGeneration.from_pretrained(flan_model_name).to(device)
    print(f"FLAN-T5-base loaded on {device}.")

In [ ]:
# --- STEP 4: RETRIEVAL FUNCTION ---

def retrieve(query, top_k=50):
    """
    Encode a query with MiniLM and retrieve the top-k most similar tweets
    from the FAISS index using cosine similarity (inner product on normalized vectors).

    Parameters:
        query (str): The search query.
        top_k (int): Number of candidate tweets to retrieve. Default 50
                     gives the reranker enough candidates to work with.

    Returns:
        list[tuple[int, float]]: List of (tweet_index, similarity_score) pairs,
                                  ordered from most to least similar.
    """
    query_vec = bi_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    # Normalize so inner product equals cosine similarity — mirrors how the index was built
    query_vec = query_vec / np.linalg.norm(query_vec)

    scores, indices = minilm_index.search(query_vec, top_k)
    return list(zip(indices[0].tolist(), scores[0].tolist()))


print("retrieve() function defined.")

In [ ]:
# --- STEP 5: RERANKING FUNCTION ---

def rerank(query, candidates, top_n=10):
    """
    Rerank a list of candidate tweets using a cross-encoder model.

    Unlike bi-encoders (which encode query and tweet independently),
    a cross-encoder reads both together and produces a single relevance score.
    This is more accurate but too slow to run on 110k tweets — hence the
    two-stage approach: FAISS narrows to ~50, cross-encoder picks the best 10.

    Parameters:
        query (str): The original search query.
        candidates (list[tuple[int, float]]): Output of retrieve() — (index, score) pairs.
        top_n (int): How many to keep after reranking.

    Returns:
        list[tuple[int, float]]: Top-n (tweet_index, cross_encoder_score) pairs,
                                  ordered by reranked relevance.
    """
    tweet_indices = [idx for idx, _ in candidates]
    candidate_texts = [tweets[idx] for idx in tweet_indices]

    # CrossEncoder expects a list of [query, passage] pairs
    pairs = [[query, text] for text in candidate_texts]
    cross_scores = cross_encoder.predict(pairs)

    # Sort by cross-encoder score descending and return top_n
    scored = sorted(zip(tweet_indices, cross_scores), key=lambda x: x[1], reverse=True)
    return scored[:top_n]


print("rerank() function defined.")

In [ ]:
# --- STEP 6: SUMMARIZATION FUNCTION ---

def summarize(query, top_tweets, max_new_tokens=200):
    """
    Use FLAN-T5 to generate a natural language summary of the retrieved tweets.

    FLAN-T5 is encoder-decoder and instruction-tuned, so it responds well
    to prompts that tell it what to do. We pass the top retrieved tweets
    as context and ask it to summarize public opinion on the query topic.

    Parameters:
        query (str): The original search query (used in the prompt).
        top_tweets (list[tuple[int, float]]): Output of rerank() — (index, score) pairs.
                                               We use the top 5 for the summary context.
        max_new_tokens (int): Maximum length of the generated summary.

    Returns:
        str: A natural language summary of what people are saying about the query.
    """
    # Use at most 5 tweets as context to keep the prompt within T5's token limit
    context_tweets = [tweets[idx] for idx, _ in top_tweets[:5]]

    numbered = '\n'.join(f"{i+1}. {t}" for i, t in enumerate(context_tweets))
    prompt = (
        f"Context tweets:\n{numbered}\n\n"
        f"Based on the tweets above, summarize what people are saying about: {query}"
    )

    inputs = flan_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    # num_beams=4: beam search produces more coherent output than greedy decoding
    output_ids = flan_model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    summary = flan_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return summary


print("summarize() function defined.")

In [ ]:
# --- STEP 7: FULL PIPELINE ---

def rag_search(query, retrieve_k=50, rerank_n=10):
    """
    Run the full end-to-end RAG pipeline: retrieve → rerank → summarize.

    Parameters:
        query (str): The natural language search query.
        retrieve_k (int): Number of candidates to pull from FAISS.
        rerank_n (int): Number of results after cross-encoder reranking.

    Returns:
        dict with keys:
            'query'     — original query string
            'retrieved' — list of (tweet_text, bi_encoder_score) before reranking
            'reranked'  — list of (tweet_text, cross_encoder_score) after reranking
            'summary'   — FLAN-T5 generated summary
    """
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")

    print(f"[1/3] Retrieving top-{retrieve_k} candidates from FAISS...")
    candidates = retrieve(query, top_k=retrieve_k)

    print(f"[2/3] Reranking with cross-encoder — keeping top-{rerank_n}...")
    reranked = rerank(query, candidates, top_n=rerank_n)

    print(f"[3/3] Summarizing top-5 with FLAN-T5...")
    summary = summarize(query, reranked)

    return {
        'query': query,
        'retrieved': [(tweets[idx], score) for idx, score in candidates[:10]],
        'reranked': [(tweets[idx], float(score)) for idx, score in reranked],
        'summary': summary
    }


print("rag_search() pipeline function defined.")

In [ ]:
# --- STEP 8: DEMO RUNS ---
# Run the full pipeline on a few example queries to see it in action.

demo_queries = [
    "I am looking for a job.",
    "climate change and the environment",
    "feeling happy and excited today"
]

results = []
for q in demo_queries:
    result = rag_search(q)
    results.append(result)

    print(f"\nTop 5 reranked tweets:")
    for i, (text, score) in enumerate(result['reranked'][:5], start=1):
        print(f"  {i}. [score={score:.3f}] {text[:120]}")

    print(f"\nFLAN-T5 Summary:")
    print(f"  {result['summary']}")
    print()

---
## Part 3 — Model Evaluation: GloVe vs MiniLM

This section benchmarks the two retrieval models against each other using standard IR metrics:
- **Precision@5** and **Precision@10** — what fraction of the top-k results are actually relevant?
- **MRR (Mean Reciprocal Rank)** — on average, how high does the first relevant tweet appear?

**Note on the test set:** The relevance judgments below were created by keyword search + manual inspection of the raw tweet data. Each query has a set of tweet indices judged relevant by inspection. In a real IR system you'd crowdsource this — for a portfolio project, 25 queries with binary relevance is standard practice (e.g. MS MARCO uses the same approach).

In [ ]:
# --- STEP 0: INSTALL DEPENDENCIES ---
!pip install faiss-cpu sentence-transformers --quiet

In [ ]:
# --- STEP 1: IMPORTS ---
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("Imports ready.")

In [ ]:
# --- STEP 2: LOAD INDEXES AND TWEETS ---

base = '/content/' if os.path.exists('/content/minilm_faiss.index') else ''

print("Loading FAISS indexes...")
minilm_index = faiss.read_index(f'{base}minilm_faiss.index')
glove_index  = faiss.read_index(f'{base}glove_faiss.index')

with open(f'{base}tweets.json', 'r', encoding='utf-8') as f:
    tweets = json.load(f)

print(f"MiniLM index: {minilm_index.ntotal:,} vectors")
print(f"GloVe index:  {glove_index.ntotal:,} vectors")
print(f"Tweets loaded: {len(tweets):,}")

In [ ]:
# --- STEP 3: LOAD ENCODERS ---

print("Loading MiniLM encoder...")
minilm_encoder = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading GloVe encoder...")
glove_encoder = SentenceTransformer('average_word_embeddings_glove.840B.300d')

print("Both encoders loaded.")

In [ ]:
# --- STEP 4: SEARCH FUNCTIONS ---

def search_minilm(query, top_k=10):
    """
    Encode a query with MiniLM and retrieve top-k tweet indices from the FAISS IP index.

    Parameters:
        query (str): The search query.
        top_k (int): Number of results to return.

    Returns:
        list[int]: Tweet indices ordered by cosine similarity (highest first).
    """
    vec = minilm_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    # Normalize to match how the index was built
    vec = vec / np.linalg.norm(vec)
    _, indices = minilm_index.search(vec, top_k)
    return indices[0].tolist()


def search_glove(query, top_k=10):
    """
    Encode a query with GloVe and retrieve top-k tweet indices from the FAISS L2 index.

    Parameters:
        query (str): The search query.
        top_k (int): Number of results to return.

    Returns:
        list[int]: Tweet indices ordered by L2 distance (closest first).
    """
    vec = glove_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    _, indices = glove_index.search(vec, top_k)
    return indices[0].tolist()


print("Search functions defined.")

In [ ]:
# --- STEP 5: TEST SET ---
#
# Each tuple: (query_string, set_of_relevant_tweet_indices)
#
# Relevance judgments were determined by keyword search + manual inspection of the
# raw tweet data. Binary relevance: 1 if the tweet is clearly about the query topic.
# Indices refer to position in tweets.json (built by Part 1).

TEST_SET = [
    ("looking for a job",
        {276, 496, 1298, 1335, 1604, 1612, 1633, 1684}),
    # All are #hiring / job-posting tweets mentioning specific openings

    ("I got fired today",
        {39145, 47191, 49915, 59279, 89940}),
    # Tweets about people being fired or laid off from jobs

    ("climate change is real",
        {1154, 3332, 4450, 5618, 8897, 9407, 10513}),
    # Tweets explicitly about climate change / global warming as a topic

    ("global warming effects",
        {1295, 14823}),
    # Tweets about specific effects of warming (drought, rising temps)
    # Intentionally small — most climate tweets in this dataset are opinion, not effects

    ("feeling happy and excited",
        {30, 1095, 1113, 2018, 2209, 2465}),
    # Tweets expressing personal happiness or excitement

    ("sad and depressed",
        {4459, 5143, 7513, 7640}),
    # Tweets about feeling sad, depressed, or heartbroken

    ("watching a movie tonight",
        {6485, 7418, 7510, 16644, 23145}),
    # Tweets about movie nights or watching films

    ("sports game results",
        {12, 134, 610, 634, 989, 2029}),
    # Tweets about playoff games, scores, and match outcomes

    ("eating delicious food",
        {2164, 4010, 7941}),
    # Tweets about food tasting good or enjoying a meal

    ("health and fitness tips",
        {2584, 4718, 9090, 9122, 14588, 15234}),
    # Tweets about gym visits, healthy eating, working out

    ("political election news",
        {132, 347, 515, 551, 749}),
    # Tweets about elections, voting, political figures

    ("new music release",
        {1897, 3243, 4662}),
    # Tweets about new songs, albums, or music playlists

    ("morning coffee routine",
        {1913, 1968, 16130, 18528, 33465, 34452}),
    # Tweets about needing or enjoying morning coffee

    ("travel plans and vacation",
        {3202, 3627, 5398, 6764, 7404}),
    # Tweets about upcoming trips, vacations, or travel plans

    ("technology and smartphones",
        {15943, 18729, 19779, 20927, 21625, 21770}),
    # Tweets about phones, iPhone/Android, and tech products

    ("social media addiction",
        {520, 2192}),
    # Tweets about social media habits — small set, topic underrepresented in dataset

    ("family and relationships",
        {106, 363, 469, 534, 794}),
    # Tweets mentioning parents, siblings, or family dynamics

    ("studying for exams",
        {936, 7605, 8317, 9634, 12779, 13927}),
    # Tweets about studying, exam stress, or cramming

    ("funny jokes and memes",
        {1005, 1354}),
    # Tweets that are clearly comedic (beyond casual lmao use)

    ("news headlines today",
        {1356, 1386, 2907, 4404}),
    # Tweets with "breaking news" or referencing current news stories

    ("feeling tired and exhausted",
        {663, 855, 1032, 1602, 1993, 2123}),
    # Tweets about tiredness, needing sleep, or feeling drained

    ("workout at the gym",
        {398, 9090, 15234, 21486, 24379, 29390, 35356}),
    # Tweets about gym sessions, leg day, cardio, lifting

    ("birthday celebration",
        {94, 578, 702, 715, 759, 951, 1015, 1700}),
    # Tweets wishing someone happy birthday or celebrating one

    ("money and financial stress",
        {1917, 7402, 8754}),
    # Tweets about being unable to afford things or financial pressure

    ("dogs and pets",
        {288, 563, 739, 1214, 4006, 6852, 7673}),
    # Tweets about dogs, cats, or pet behavior
]

print(f"Test set: {len(TEST_SET)} queries, all relevance judgments filled in.")
print(f"Total judged-relevant tweets: {sum(len(r) for _, r in TEST_SET)}")
print("Ready to run evaluation — proceed to the metric cells.")

In [ ]:
# --- HELPER: INSPECT RESULTS FOR A QUERY ---
# Use this cell to browse top results and decide which indices to mark as relevant.
# Change the query and run to inspect different topics.

INSPECT_QUERY = "looking for a job"
TOP_N = 20

minilm_results = search_minilm(INSPECT_QUERY, top_k=TOP_N)
glove_results  = search_glove(INSPECT_QUERY, top_k=TOP_N)

print(f"Query: '{INSPECT_QUERY}'")
print(f"\n{'MiniLM Results':^60} | {'GloVe Results':^60}")
print("-" * 130)

for i in range(TOP_N):
    m_idx = minilm_results[i]
    g_idx = glove_results[i]
    m_text = tweets[m_idx][:55].replace('\n', ' ')
    g_text = tweets[g_idx][:55].replace('\n', ' ')
    print(f"  [{m_idx:6d}] {m_text:<55} | [{g_idx:6d}] {g_text}")

In [ ]:
# --- STEP 6: METRIC FUNCTIONS ---

def precision_at_k(retrieved, relevant, k):
    """
    Compute Precision@k: the fraction of the top-k results that are relevant.

    Parameters:
        retrieved (list[int]): Ranked list of retrieved tweet indices.
        relevant (set[int]): Set of tweet indices judged relevant for this query.
        k (int): Cutoff rank.

    Returns:
        float: Precision@k score in [0, 1]. Returns 0 if relevant set is empty.
    """
    if not relevant:
        return 0.0
    top_k = retrieved[:k]
    hits = sum(1 for idx in top_k if idx in relevant)
    return hits / k


def reciprocal_rank(retrieved, relevant):
    """
    Compute the Reciprocal Rank for a single query: 1/rank of the first relevant result.

    If no relevant tweet appears in the retrieved list, returns 0.
    MRR averages this across all queries — a higher MRR means relevant results
    appear closer to the top on average.

    Parameters:
        retrieved (list[int]): Ranked list of retrieved tweet indices.
        relevant (set[int]): Set of tweet indices judged relevant.

    Returns:
        float: Reciprocal rank score in (0, 1], or 0 if no relevant result found.
    """
    for rank, idx in enumerate(retrieved, start=1):
        if idx in relevant:
            return 1.0 / rank
    return 0.0


print("Metric functions defined.")

In [ ]:
# --- STEP 7: RUN EVALUATION ---
# Skip queries with empty relevance sets (not yet judged)

judged_queries = [(q, rel) for q, rel in TEST_SET if len(rel) > 0]

if len(judged_queries) == 0:
    print("No relevance judgments found in TEST_SET yet.")
    print("Use the inspection helper cell above to browse tweets and fill in the relevant indices.")
else:
    minilm_p5, minilm_p10, minilm_rr = [], [], []
    glove_p5,  glove_p10,  glove_rr  = [], [], []

    print(f"Evaluating {len(judged_queries)} judged queries...")

    for query, relevant in judged_queries:
        m_results = search_minilm(query, top_k=10)
        g_results = search_glove(query, top_k=10)

        minilm_p5.append(precision_at_k(m_results, relevant, 5))
        minilm_p10.append(precision_at_k(m_results, relevant, 10))
        minilm_rr.append(reciprocal_rank(m_results, relevant))

        glove_p5.append(precision_at_k(g_results, relevant, 5))
        glove_p10.append(precision_at_k(g_results, relevant, 10))
        glove_rr.append(reciprocal_rank(g_results, relevant))

    print("Evaluation complete.")

In [ ]:
# --- STEP 8: PRINT COMPARISON TABLE ---

if len(judged_queries) == 0:
    print("No results to display — fill in TEST_SET first.")
else:
    def avg(lst):
        return sum(lst) / len(lst) if lst else 0.0

    m_p5  = avg(minilm_p5)
    m_p10 = avg(minilm_p10)
    m_mrr = avg(minilm_rr)
    g_p5  = avg(glove_p5)
    g_p10 = avg(glove_p10)
    g_mrr = avg(glove_rr)

    # Compute relative improvement of MiniLM over GloVe
    def pct_diff(a, b):
        """Percentage difference of a over b. Returns 'N/A' if b is zero."""
        return f"+{(a - b) / b * 100:.1f}%" if b > 0 else "N/A"

    print(f"\n{'Metric':<15} {'MiniLM':>10} {'GloVe':>10} {'MiniLM vs GloVe':>18}")
    print("-" * 55)
    print(f"{'Precision@5':<15} {m_p5:>10.4f} {g_p5:>10.4f} {pct_diff(m_p5, g_p5):>18}")
    print(f"{'Precision@10':<15} {m_p10:>10.4f} {g_p10:>10.4f} {pct_diff(m_p10, g_p10):>18}")
    print(f"{'MRR':<15} {m_mrr:>10.4f} {g_mrr:>10.4f} {pct_diff(m_mrr, g_mrr):>18}")
    print(f"\nEvaluated on {len(judged_queries)} queries.")

    if m_mrr > g_mrr:
        print(f"\nResult: MiniLM outperformed GloVe by {(m_mrr - g_mrr) / g_mrr * 100:.1f}% on MRR.")
    elif g_mrr > m_mrr:
        print(f"\nResult: GloVe outperformed MiniLM by {(g_mrr - m_mrr) / m_mrr * 100:.1f}% on MRR.")
    else:
        print("\nResult: Both models achieved identical MRR.")

In [ ]:
# --- BONUS: PER-QUERY BREAKDOWN ---
# Useful for seeing which query types favor MiniLM vs GloVe

if len(judged_queries) > 0:
    print(f"\n{'Query':<40} {'MiniLM MRR':>12} {'GloVe MRR':>12} {'Winner':>10}")
    print("-" * 78)
    for i, (query, _) in enumerate(judged_queries):
        m = minilm_rr[i]
        g = glove_rr[i]
        winner = 'MiniLM' if m > g else ('GloVe' if g > m else 'Tie')
        print(f"  {query[:38]:<40} {m:>12.4f} {g:>12.4f} {winner:>10}")

---
## Part 4 — UMAP Embedding Visualization

This section projects the MiniLM tweet embeddings from 384 dimensions down to 2D using **UMAP**, then clusters them with **K-Means** to reveal topical structure in the tweet dataset.

**What UMAP does:** It finds a 2D layout that preserves the local neighborhood structure of the high-dimensional space. Tweets that are semantically similar end up near each other in the plot.

**What to expect:** Distinct islands in the scatter plot correspond to topic clusters — job postings, sports, politics, etc. The cluster sample cell at the end prints example tweets from each group so you can interpret what each island represents.

In [ ]:
# --- STEP 0: INSTALL DEPENDENCIES ---
# umap-learn: the UMAP algorithm
# scikit-learn: K-Means clustering
# matplotlib: plotting
!pip install umap-learn scikit-learn matplotlib --quiet

In [ ]:
# --- STEP 1: IMPORTS ---
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import umap
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

print("Imports ready.")

In [ ]:
# --- STEP 2: LOAD MINILM EMBEDDINGS ---

base = '/content/' if os.path.exists('/content/minilm_embeddings.npy') else ''

print("Loading MiniLM embeddings...")
embeddings = np.load(f'{base}minilm_embeddings.npy')
print(f"Embeddings shape: {embeddings.shape}  ({embeddings.shape[0]:,} tweets \u00d7 {embeddings.shape[1]} dims)")

with open(f'{base}tweets.json', 'r', encoding='utf-8') as f:
    tweets = json.load(f)

In [ ]:
# --- STEP 3: SAMPLE FOR SPEED (OPTIONAL) ---
# UMAP on 110k × 384 can take 10-20 minutes on CPU.
# We sample 20k tweets — enough to reveal cluster structure clearly.
# Remove the sampling to plot everything (expect longer runtime).

SAMPLE_SIZE = 20_000

if len(embeddings) > SAMPLE_SIZE:
    # Use a fixed seed so the sample is reproducible across runs
    rng = np.random.default_rng(seed=42)
    sample_idx = rng.choice(len(embeddings), size=SAMPLE_SIZE, replace=False)
    sample_idx.sort()
    emb_sample = embeddings[sample_idx]
    tweet_sample = [tweets[i] for i in sample_idx]
    print(f"Sampled {SAMPLE_SIZE:,} of {len(embeddings):,} tweets for visualization.")
else:
    emb_sample = embeddings
    tweet_sample = tweets
    print(f"Using all {len(embeddings):,} tweets (no sampling needed).")

In [ ]:
# --- STEP 4: UMAP DIMENSIONALITY REDUCTION ---

def run_umap(embeddings, n_neighbors=15, min_dist=0.1, random_state=42):
    """
    Project high-dimensional embeddings to 2D using UMAP.

    Parameters:
        embeddings (np.ndarray): Float array of shape (n, dim).
        n_neighbors (int): Controls how much local vs. global structure UMAP preserves.
                           Lower = more local clusters; higher = broader structure.
        min_dist (float): Minimum distance between points in 2D space.
                          Lower = tighter clusters; higher = more spread out.
        random_state (int): Seed for reproducibility.

    Returns:
        np.ndarray: 2D array of shape (n, 2).
    """
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric='cosine',  # use cosine distance — matches how we built the index
        random_state=random_state
    )
    return reducer.fit_transform(embeddings)


print("Running UMAP (this takes a few minutes on CPU)...")
embedding_2d = run_umap(emb_sample)
print(f"UMAP complete — output shape: {embedding_2d.shape}")

In [ ]:
# --- STEP 5: K-MEANS CLUSTERING ---

def cluster_embeddings(embeddings_2d, n_clusters=12, random_state=42):
    """
    Cluster the 2D UMAP projections with K-Means.

    We run K-Means on the 2D projections (not the original high-dim embeddings)
    because the goal is to color the visualization — the 2D layout already
    encodes semantic proximity, so clustering in 2D gives meaningful groups.

    Parameters:
        embeddings_2d (np.ndarray): 2D UMAP output, shape (n, 2).
        n_clusters (int): Number of topic clusters. 10-15 works well for tweet data.
        random_state (int): Seed for reproducibility.

    Returns:
        np.ndarray: Integer cluster label for each point.
    """
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init='auto')
    labels = kmeans.fit_predict(embeddings_2d)
    return labels


N_CLUSTERS = 12
print(f"Clustering into {N_CLUSTERS} groups with K-Means...")
cluster_labels = cluster_embeddings(embedding_2d, n_clusters=N_CLUSTERS)
print(f"Clustering complete. Label distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Cluster {label}: {count:,} tweets")

In [ ]:
# --- STEP 6: PLOT AND SAVE ---

def plot_umap_clusters(embedding_2d, labels, n_clusters, output_path='umap_clusters.png'):
    """
    Create and save a scatter plot of UMAP-projected tweet embeddings, colored by cluster.

    Parameters:
        embedding_2d (np.ndarray): 2D UMAP coordinates, shape (n, 2).
        labels (np.ndarray): Cluster label for each point.
        n_clusters (int): Total number of clusters (used for colormap selection).
        output_path (str): File path to save the PNG figure.

    Returns:
        None. Saves figure to output_path and displays inline.
    """
    fig, ax = plt.subplots(figsize=(14, 10))

    # Use a colormap with enough distinct colors for all clusters
    cmap = plt.cm.get_cmap('tab20', n_clusters)

    scatter = ax.scatter(
        embedding_2d[:, 0],
        embedding_2d[:, 1],
        c=labels,
        cmap=cmap,
        s=1.5,        # small dots so 20k points don't overlap too much
        alpha=0.6     # slight transparency to show density
    )

    cbar = plt.colorbar(scatter, ax=ax, ticks=range(n_clusters))
    cbar.set_label('Cluster', fontsize=12)

    ax.set_title(
        f'UMAP Projection of {len(embedding_2d):,} Tweet Embeddings\n'
        f'(MiniLM all-MiniLM-L6-v2, {n_clusters} K-Means clusters)',
        fontsize=14
    )
    ax.set_xlabel('UMAP Dimension 1', fontsize=12)
    ax.set_ylabel('UMAP Dimension 2', fontsize=12)
    ax.set_facecolor('#f8f8f8')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure saved to {output_path}")


plot_umap_clusters(embedding_2d, cluster_labels, N_CLUSTERS)

In [ ]:
# --- STEP 7: SAMPLE TWEETS PER CLUSTER ---
# Print 3 example tweets from each cluster so we can understand what each group represents.
# Useful for labeling clusters in a README or portfolio description.

SAMPLES_PER_CLUSTER = 3

print(f"Sample tweets from each cluster (to help interpret the visualization):")
print("=" * 70)

for cluster_id in range(N_CLUSTERS):
    # Get indices of all tweets in this cluster
    cluster_mask = np.where(cluster_labels == cluster_id)[0]

    # Pick a few examples without replacement
    chosen = np.random.choice(cluster_mask, size=min(SAMPLES_PER_CLUSTER, len(cluster_mask)), replace=False)

    print(f"\nCluster {cluster_id} ({len(cluster_mask):,} tweets):")
    for idx in chosen:
        print(f"  - {tweet_sample[idx][:110].replace(chr(10), ' ')}")

---
## Part 5 — Interactive Gradio Demo

This section wraps the full retrieval pipeline in a **Gradio** web UI so anyone can try it without touching code.

**Features:**
- Text input for the search query
- Dropdown to choose retrieval model: `GloVe`, `MiniLM`, or `MiniLM + Cross-encoder Reranker`
- Slider for how many tweets to return (top-k)
- Output: retrieved tweets + FLAN-T5 generated summary

**Deployment:** The last cell launches with `share=True`, which creates a public tunnel URL that works from Colab. You can also deploy permanently to HuggingFace Spaces by exporting this section as a standalone `app.py`.

In [ ]:
# --- STEP 0: INSTALL DEPENDENCIES ---
!pip install gradio faiss-cpu sentence-transformers transformers torch --quiet

In [ ]:
# --- STEP 1: IMPORTS ---
import json
import os
import numpy as np
import faiss
import torch
import gradio as gr
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# --- STEP 2: LOAD INDEXES AND TWEETS ---

base = '/content/' if os.path.exists('/content/minilm_faiss.index') else ''

print("Loading FAISS indexes...")
minilm_index = faiss.read_index(f'{base}minilm_faiss.index')
glove_index  = faiss.read_index(f'{base}glove_faiss.index')

with open(f'{base}tweets.json', 'r', encoding='utf-8') as f:
    tweets = json.load(f)

print(f"MiniLM index: {minilm_index.ntotal:,} vectors")
print(f"GloVe index:  {glove_index.ntotal:,} vectors")
print(f"Tweets loaded: {len(tweets):,}")

In [ ]:
# --- STEP 3: LOAD MODELS ---

print("Loading MiniLM bi-encoder...")
minilm_encoder = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading GloVe encoder...")
glove_encoder = SentenceTransformer('average_word_embeddings_glove.840B.300d')

print("Loading cross-encoder reranker...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("Loading FLAN-T5 for summarization...")
try:
    flan_name = 'google/flan-t5-large'
    flan_tokenizer = T5Tokenizer.from_pretrained(flan_name)
    flan_model = T5ForConditionalGeneration.from_pretrained(flan_name).to(device)
    print("FLAN-T5-large loaded.")
except RuntimeError:
    # Fall back to base if large doesn't fit in memory
    print("FLAN-T5-large OOM — loading flan-t5-base instead.")
    flan_name = 'google/flan-t5-base'
    flan_tokenizer = T5Tokenizer.from_pretrained(flan_name)
    flan_model = T5ForConditionalGeneration.from_pretrained(flan_name).to(device)
    print("FLAN-T5-base loaded.")

print("\nAll models ready!")

In [ ]:
# --- STEP 4: PIPELINE FUNCTIONS ---

def retrieve_minilm(query, top_k):
    """
    Retrieve top-k tweets using MiniLM + FAISS cosine search.

    Parameters:
        query (str): Search query.
        top_k (int): Number of results to return.

    Returns:
        list[int]: Tweet indices ordered by similarity (best first).
    """
    vec = minilm_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    vec = vec / np.linalg.norm(vec)  # normalize for cosine similarity via inner product
    _, indices = minilm_index.search(vec, top_k)
    return indices[0].tolist()


def retrieve_glove(query, top_k):
    """
    Retrieve top-k tweets using GloVe averaged embeddings + FAISS L2 search.

    Parameters:
        query (str): Search query.
        top_k (int): Number of results to return.

    Returns:
        list[int]: Tweet indices ordered by L2 distance (closest first).
    """
    vec = glove_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    _, indices = glove_index.search(vec, top_k)
    return indices[0].tolist()


def rerank(query, tweet_indices, top_n):
    """
    Rerank candidate tweets with a cross-encoder and return the top-n.

    Parameters:
        query (str): Original search query.
        tweet_indices (list[int]): Candidate tweet indices from FAISS retrieval.
        top_n (int): How many to keep after reranking.

    Returns:
        list[int]: Top-n tweet indices in reranked order.
    """
    pairs = [[query, tweets[idx]] for idx in tweet_indices]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(tweet_indices, scores), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in ranked[:top_n]]


def generate_summary(query, tweet_indices):
    """
    Generate a FLAN-T5 summary of the top retrieved tweets.

    Parameters:
        query (str): The original search query.
        tweet_indices (list[int]): Tweet indices to use as context (top 5 used).

    Returns:
        str: Natural language summary generated by FLAN-T5.
    """
    context = [tweets[i] for i in tweet_indices[:5]]
    numbered = '\n'.join(f"{i+1}. {t}" for i, t in enumerate(context))
    prompt = (
        f"Context tweets:\n{numbered}\n\n"
        f"Based on the tweets above, summarize what people are saying about: {query}"
    )
    inputs = flan_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    out = flan_model.generate(**inputs, max_new_tokens=200, num_beams=4)
    return flan_tokenizer.decode(out[0], skip_special_tokens=True)


print("Pipeline functions defined.")

In [ ]:
# --- STEP 5: GRADIO HANDLER ---

def search_handler(query, model_choice, top_k):
    """
    Main Gradio handler — runs the selected pipeline and returns formatted results.

    Parameters:
        query (str): User's search query from the text box.
        model_choice (str): One of 'GloVe', 'MiniLM', or 'MiniLM + Reranker'.
        top_k (int): Number of tweets to display (from the slider).

    Returns:
        tuple[str, str]: (formatted tweet results, FLAN-T5 summary)
    """
    if not query.strip():
        return "Please enter a query.", ""

    top_k = int(top_k)

    if model_choice == 'GloVe':
        indices = retrieve_glove(query, top_k)

    elif model_choice == 'MiniLM':
        indices = retrieve_minilm(query, top_k)

    else:  # MiniLM + Reranker
        # Retrieve a larger pool first so the reranker has more candidates to work with
        candidates = retrieve_minilm(query, top_k=50)
        indices = rerank(query, candidates, top_n=top_k)

    summary = generate_summary(query, indices)

    # Format tweet list for display
    tweet_lines = []
    for rank, idx in enumerate(indices, start=1):
        tweet_lines.append(f"{rank}. {tweets[idx]}")
    tweet_display = "\n\n".join(tweet_lines)

    return tweet_display, summary


print("Gradio handler defined.")

In [ ]:
# --- STEP 6: LAUNCH GRADIO INTERFACE ---

demo = gr.Interface(
    fn=search_handler,
    inputs=[
        gr.Textbox(
            label="Search Query",
            placeholder="e.g. looking for a job",
            lines=1
        ),
        gr.Dropdown(
            choices=['GloVe', 'MiniLM', 'MiniLM + Reranker'],
            value='MiniLM + Reranker',
            label="Retrieval Model"
        ),
        gr.Slider(
            minimum=5,
            maximum=25,
            value=10,
            step=1,
            label="Number of tweets (top-k)"
        )
    ],
    outputs=[
        gr.Textbox(label="Retrieved Tweets", lines=15),
        gr.Textbox(label="FLAN-T5 Summary", lines=5)
    ],
    title="SpatialSearch — Tweet RAG Pipeline",
    description=(
        "Search 110,000+ tweets using semantic embeddings. "
        "Choose between GloVe (static embeddings), MiniLM (contextual BERT-based), "
        "or MiniLM + a cross-encoder reranker for highest accuracy. "
        "FLAN-T5 summarizes the top results."
    ),
    examples=[
        ["looking for a job",       "MiniLM + Reranker", 10],
        ["climate change",           "MiniLM",            10],
        ["feeling happy today",      "GloVe",             10],
    ],
    allow_flagging='never'
)

# share=True creates a public tunnel URL (works on Colab and local)
# Set share=False to run locally only
demo.launch(share=True)